# XAI IDS Benchmark - Google Colab Runner

## Setup Instructions
1. Upload `xai_ids_benchmark/` project folder to Google Drive
2. Upload `CIC IoT-DIAD 2024.zip` dataset to Google Drive
3. Your Google Drive structure:
```
My Drive/
  xai_ids_benchmark/
    main.py
    src/
    colab_runner.ipynb
  CIC IoT-DIAD 2024.zip   (or inside xai_ids_benchmark/)
```
4. Runtime > Change runtime type > **T4 GPU**
5. Run all cells in order

## Batch Execution (if Colab disconnects)
Colab sessions last ~12 hours. If the pipeline doesn't finish:
- All results are auto-saved to Google Drive after each stage
- Just change `STAGES` in **Cell 7** and re-run all cells
- Stage 1 loads from cache (~2 min instead of 25 min)
- Missing dependencies (Stage 2, 3) are auto-loaded when needed

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set Project Path & Extract Dataset
Change paths below if your folders are in different locations.

In [ ]:
import os
import zipfile
import glob

# === CHANGE THESE if your files are in different Drive folders ===
PROJECT_PATH = '/content/drive/MyDrive/xai_ids_benchmark'
# Where to look for the dataset zip (searches multiple locations)
ZIP_SEARCH_PATHS = [
    '/content/drive/MyDrive/CIC IoT-DIAD 2024.zip',
    '/content/drive/MyDrive/xai_ids_benchmark/CIC IoT-DIAD 2024.zip',
    '/content/drive/MyDrive/CIC_IoT-DIAD_2024.zip',
    '/content/drive/MyDrive/dataset.zip',
]

assert os.path.exists(PROJECT_PATH), f"Project not found at {PROJECT_PATH}"
os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

# --- Extract dataset zip if not already extracted ---
dataset_dir = os.path.join(PROJECT_PATH, 'CIC IoT-DIAD 2024')

if os.path.exists(dataset_dir) and os.listdir(dataset_dir):
    print(f"Dataset already extracted at: {dataset_dir}")
else:
    # Find the zip file
    zip_path = None
    for p in ZIP_SEARCH_PATHS:
        if os.path.exists(p):
            zip_path = p
            break

    # Also search for any zip containing "CIC" or "IoT" in Drive root and project
    if zip_path is None:
        for search_dir in ['/content/drive/MyDrive', PROJECT_PATH]:
            for f in os.listdir(search_dir):
                if f.endswith('.zip') and ('CIC' in f or 'IoT' in f or 'DIAD' in f):
                    zip_path = os.path.join(search_dir, f)
                    break
            if zip_path:
                break

    if zip_path:
        print(f"Found dataset zip: {zip_path}")
        zip_size = os.path.getsize(zip_path) / (1024**3)
        print(f"Size: {zip_size:.2f} GB")
        print("Extracting to project directory (this may take a few minutes)...")

        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(PROJECT_PATH)

        print("Extraction complete!")

        # Check if extraction created nested folder or direct
        if not os.path.exists(dataset_dir):
            # Maybe zip has different folder name - find it
            extracted = [d for d in os.listdir(PROJECT_PATH)
                        if os.path.isdir(os.path.join(PROJECT_PATH, d))
                        and ('CIC' in d or 'IoT' in d or 'DIAD' in d)]
            if extracted:
                actual = os.path.join(PROJECT_PATH, extracted[0])
                os.rename(actual, dataset_dir)
                print(f"Renamed '{extracted[0]}' -> 'CIC IoT-DIAD 2024'")
    else:
        print("ERROR: Dataset zip not found!")
        print("Searched:", ZIP_SEARCH_PATHS)
        print("\nPlease upload your dataset zip to Google Drive and update ZIP_SEARCH_PATHS above")

print(f"\nProject files: {os.listdir('.')}")

## 3. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 4. Install Dependencies

In [ ]:
!pip install -q xgboost lightgbm catboost shap lime anchor-exp imbalanced-learn statsmodels

## 5. Verify Dataset

In [ ]:
dataset_path = os.path.join(PROJECT_PATH, 'CIC IoT-DIAD 2024')
if os.path.exists(dataset_path):
    folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    print(f"Dataset found: {len(folders)} class folders")
    for f in sorted(folders):
        csvs = [c for c in os.listdir(os.path.join(dataset_path, f)) if c.endswith('.csv')]
        print(f"  {f}: {len(csvs)} CSV files")
else:
    print(f"ERROR: Dataset not found at {dataset_path}")
    print("Please upload 'CIC IoT-DIAD 2024' folder inside your project folder on Google Drive")

## 6. Add src/ to Python path

In [ ]:
import sys
src_path = os.path.join(PROJECT_PATH, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"Added to path: {src_path}")

# Quick import test
from config import RANDOM_SEED, RESULTS_DIR
print(f"Config loaded. RANDOM_SEED={RANDOM_SEED}")

## 7. Run Pipeline

### Configuration
- `MODE`: `'multiclass'` (8 classes) or `'binary'` (Benign vs Attack)
- `STAGES`: Which stages to run. Examples:
  - `[1,2,3,4,5,6,7,8,9,10]` — All stages (first time)
  - `[4,5,6]` — Only stages 4-6 (Stage 1 auto-loads from cache, Stage 2/3 auto-run if needed)
  - `[7,8,9,10]` — Only stages 7-10
- `QUICK`: `True` = fewer epochs/folds (for testing), `False` = full results

In [ ]:
# === CONFIGURATION ===
MODE = 'multiclass'   # 'multiclass' or 'binary'
STAGES = [1,2,3,4,5,6,7,8,9,10]  # which stages to run
QUICK = False          # True = faster (fewer epochs, fewer folds)

### Run All Selected Stages
This cell runs all stages listed in `STAGES` above.
- Stage 1 always runs first (uses disk cache if available — fast reload)
- If Stage 5/6/8/9 needs Stage 3 models but Stage 3 is not in `STAGES`, it auto-trains them
- If Stage 4 needs Stage 2 features but Stage 2 is not in `STAGES`, it auto-runs it
- All results + visualizations + LaTeX tables are saved to `results/{MODE}/`

In [ ]:
import importlib
import time
import main as pipeline
importlib.reload(pipeline)

print(f"Mode: {MODE}")
print(f"Stages: {STAGES}")
print(f"Quick: {QUICK}")
print("=" * 60)

start_time = time.time()

# _run_pipeline_for_mode handles ALL dependencies automatically:
# - Always loads Stage 1 first (from cache if available)
# - Auto-runs Stage 2 if Stage 4 needs it
# - Auto-runs Stage 3 if Stage 5/9 needs it
# - Generates visualizations, error analysis, ablation, LaTeX tables
all_results = pipeline._run_pipeline_for_mode(
    mode=MODE,
    quick=QUICK,
    stages=STAGES
)

elapsed = time.time() - start_time
print("=" * 60)
print(f"DONE! Total time: {elapsed/3600:.1f} hours ({elapsed:.0f} seconds)")
print(f"Results saved to: results/{MODE}/")

## 8. Check Results

In [ ]:
import glob
results_dir = os.path.join(PROJECT_PATH, 'results', MODE)
if os.path.exists(results_dir):
    for stage_dir in sorted(os.listdir(results_dir)):
        full_path = os.path.join(results_dir, stage_dir)
        if os.path.isdir(full_path):
            files = os.listdir(full_path)
            print(f"{stage_dir}: {len(files)} files")
            for f in sorted(files)[:5]:
                size = os.path.getsize(os.path.join(full_path, f)) / 1024
                print(f"  {f} ({size:.1f} KB)")
else:
    print(f"No results yet at {results_dir}")

## 9. Download Results (Optional)
Download the results folder as a zip file.

In [ ]:
import shutil
from google.colab import files

results_dir = os.path.join(PROJECT_PATH, 'results')
if os.path.exists(results_dir):
    zip_path = '/content/xai_ids_results'
    shutil.make_archive(zip_path, 'zip', results_dir)
    files.download(zip_path + '.zip')
    print("Downloading results.zip...")
else:
    print("No results to download.")